### Learning Instruction Finetuning

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import os
import socket
import sys
from dotenv import load_dotenv
from pathlib import Path
import sys

In [3]:
try:
    project_root = next(p for p in Path().cwd().resolve().parents if (p /"env.env").exists())
except StopIteration:
    # Fallback to current working directory if not found
    project_root = Path.cwd()

sys.path.insert(0, str(project_root))

print(f"Project Root: {project_root}")
load_dotenv((project_root /'env.env').resolve())
data_dir = project_root / "datasets"

Project Root: /storage/ice1/1/0/vchopra37/projects/learning_env


In [4]:
# Check if running on the Pace server
hostname = socket.gethostname()
IS_PACE_SERVER = "pace" in hostname.lower()  # Replace 'pace' with a unique keyword from your server's hostname
IS_PACE_SERVER

True

In [5]:
import torch
from datasets import Dataset as hf_data
from torch.utils.data import Dataset
import tiktoken
import numpy, tqdm
import matplotlib.pyplot as plt
%matplotlib inline

In [6]:
## Accelerator for Pytorch
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device

'cuda'

In [ ]:
# MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

In [8]:

from model_training.framework.data.post_train.instruction_tuning.raw_data.simple_example.instruct_data import download_and_load_file
from model_training.framework.data.post_train.instruction_tuning.prompt_style.styles import SFTDatasetPrepare, AlpacaFormatOld
from model_training.framework.data.post_train.dataloader import InstructionDataset, custom_collate_fn, create_instruction_dataloader

    ## Fix this import
    
    # raise Exception("Not Running on Pace")
        # from src.model import GPT

In [ ]:
! pip install accelerate

In [ ]:
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(i, props.name, f"{props.total_memory / 1024**3:.1f} GiB")

### Step-1:-  Preparing the dataset

We are using Alpaca Format, every models have their own format, so the format of the tuning can differ. <br>
THis format is called ALPACA format{instruction, input output}... <br>
 There are different formats that exist and the data format doesn't matter, we need to take prompt style templates for instruction tuning. We need to apply those prompt templates for instruction tuning. 

In [9]:
file_name = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)
data = download_and_load_file(data_dir=(data_dir / "instruction_tuning"), file_name=file_name, url = url)
print("Number of entries:", len(data))
data[50] , data[999]

Number of entries: 1100


({'instruction': 'Identify the correct spelling of the following word.',
  'input': 'Ocassion',
  'output': "The correct spelling is 'Occasion.'"},
 {'instruction': "What is an antonym of 'complicated'?",
  'input': '',
  'output': "An antonym of 'complicated' is 'simple'."})

In [10]:
# alpaca = AlpacaFormatOld(data)

In [11]:
# ## Example of format
# print(alpaca.format_input(instruction=data[50]['instruction'], in_text=data[50]['input']))
# print("=============================================")
# print(alpaca.detailed_response(out_text = data[50]['output']))

In [12]:
# sample = (alpaca.complete_response(instruction=data[999]['instruction'], in_text=data[999]['input'], out_text=data[999]['output']))
# print(sample)

In [13]:
dataset  = hf_data.from_list(data)
sft_data= SFTDatasetPrepare(model_name=MODEL_NAME, template="model",)
prepared = sft_data.prepare(dataset)

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

In [14]:
print(prepared[0])
print(sft_data.preview(prepared[0]))

{'prompt': [{'content': 'Evaluate the following phrase by transforming it into the spelling given.\n\nInput:\nfreind --> friend', 'role': 'user'}], 'completion': [{'content': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".', 'role': 'assistant'}]}
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Evaluate the following phrase by transforming it into the spelling given.

Input:
freind --> friend<|im_end|>
<|im_start|>assistant
The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".<|im_end|>



#### Spliting the data 

In [15]:
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)    # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation

train_data = prepared.select(range(train_portion))
test_data = prepared.select(
    range(train_portion, train_portion + test_portion)
)
val_data = prepared.select(
    range(train_portion + test_portion, len(prepared))
)

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


### Step-2: - Prepare the Training Set Batches

#### Setting up Tokenizer

In [16]:
# tokenizer = tiktoken.get_encoding("gpt2")
# pad_token_id= tokenizer.encode('<|endoftext|>', allowed_special={"<|endoftext|>"})[0]
# pad_token_id
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

pad_token_id = tokenizer.pad_token_id
pad_token_id

151643

In [17]:
train_dataset = InstructionDataset(
    ds=train_data,
    tokenizer=tokenizer,
    prompt_style=sft_data,
)


In [18]:
val_dataset = InstructionDataset(
    ds=val_data,
    tokenizer=tokenizer,
    prompt_style=sft_data,
)
test_dataset = InstructionDataset(
    ds=test_data,
    tokenizer=tokenizer,
    prompt_style=sft_data,
)

#### Sample Data Prepare

In [19]:

inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (
    inputs_1,
    inputs_2,
    inputs_3
)
custom_collate_fn(batch, pad_token_id=pad_token_id)

(tensor([[     0,      1,      2,      3,      4],
         [     5,      6, 151643, 151643, 151643],
         [     7,      8,      9, 151643, 151643]]),
 tensor([[     1,      2,      3,      4, 151643],
         [     6, 151643,   -100,   -100,   -100],
         [     8,      9, 151643,   -100,   -100]]))

In [20]:
logits_1 = torch.tensor(
    [[-1.0, 1.0],  # 1st training example
     [-0.5, 1.5]]  # 2nd training example
)
targets_1 = torch.tensor([0, 1])


loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
print(loss_1)

logits_2 = torch.tensor(
    [[-1.0, 1.0],
     [-0.5, 1.5],
     [-0.5, 1.5]]  # New 3rd training example
)
targets_2 = torch.tensor([0, 1, 1])

loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)
targets_3 = torch.tensor([0, 1, -100])

loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)
print("loss_1 == loss_3:", loss_1 == loss_3)

tensor(1.1269)
tensor(0.7936)
tensor(1.1269)
loss_1 == loss_3: tensor(True)


### Creating, Training and Testing Dataset

In [21]:
from functools import partial
# Changing the defaults at runtime
custom_collate_fn1 = partial(custom_collate_fn, device=device, allowed_max_length = 1024, pad_token_id=pad_token_id)

In [22]:
from torch.utils.data import DataLoader


num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_loader = create_instruction_dataloader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)
val_loader = create_instruction_dataloader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_loader = create_instruction_dataloader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

In [23]:
print("Train loader:")
for inputs, targets in train_loader:
    sample_input , sample_target = inputs, targets
    print(inputs.shape, targets.shape)
    break

Train loader:
torch.Size([8, 59]) torch.Size([8, 59])


### Step-4: - Loading the Model Testing inference

In [24]:
from model_training.framework.model.load_model.gpt2.model import return_model
from model_training.framework.model.load_model.gpt2.configs import load_model_config
from model_training.framework.model.components.inference.inference import gen_sample
from model_training.framework.model.tokenizer.tokenizer_utils import text_to_token_ids, token_ids_to_text

In [25]:
config =load_model_config(dataset_max_len=1024)
# MODEL_NAME = 'gpt2'

model = return_model(MODEL_NAME)

# model = GPT.from_pretrained(config, "gpt2")
# print(f"Model Loaded: {model}")
# model.eval()
# model = model.to(device)

loading wights from pretrained gpt: gpt2-small
{'n_layer': 12, 'n_head': 12, 'emb_dim': 768}


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [26]:
# import inspect
# inspect.signature(tokenizer.encode)

In [27]:
model.eval()
text_1 = "Every effort moves you"
in_tokens = text_to_token_ids(text_1, tokenizer=tokenizer, 
                            #   allowed_special = {"<|endoftext|>"}
                              )
in_tokens = in_tokens.to(next(model.parameters()).device)
print(in_tokens.shape)  # [1, sequence_length]

torch.Size([1, 4])


In [28]:
out_tokens = gen_sample(model, in_tokens, max_new_tokens=40, context_len=1024)
print(token_ids_to_text(out_tokens, tokenizer))

Every effort moves you closer towards a goal. Think of goals in terms of steps: the ultimate goal may be the top of an enormous mountain, but the very first thing you do to start climbing is take your first step


### SFT-Training

In [29]:
import torch.nn.functional as F

model = model.to(device).eval()
sample_input = sample_input.to(device)
sample_target = sample_target.to(device)

with torch.no_grad():
    out = model(input_ids=sample_input)
    loss = F.cross_entropy(
        out.logits.reshape(-1, out.logits.size(-1)),
        sample_target.reshape(-1),
        ignore_index=-100,
    )

print(loss.item())

4.09375


In [32]:
from model_training.framework.train.core.train import train_model
from model_training.framework.train.core.config import TrainConfig
from model_training.framework.train.utils.optimizer import configure_optimizer
from model_training.framework.train.utils.lr_scheduler import cosine_lr

In [38]:
seq_length = 1024
total_batch_size = batch_size * seq_length
micro_batch_size = 4
grad_accum_steps = total_batch_size // (micro_batch_size * seq_length )
# seq_length = 1024
# micro_batch_size = train_loader.batch_size  # 1
# grad_accum_steps = 8
num_epochs = 2

In [39]:
from dataclasses import  dataclass
from typing import Any

@dataclass
class TrainComponents:
    optimizer:Any
    dataloader:Any
    scheduler:Any
    

In [40]:


train_config = TrainConfig(
    num_steps=(num_epochs * len(train_loader.loader)) // grad_accum_steps,
    learning_rate=1e-5,
    total_batch_size=micro_batch_size * seq_length * grad_accum_steps,
    micro_batch_size=micro_batch_size,
    seq_length=seq_length,
    accumulation_steps=grad_accum_steps,
    max_lr=1e-5,
    min_lr=1e-5,
    warmup_steps=0,
    weight_decay=0.1,
    device=device,
    use_compile=False,
    precision="fp32",
)

In [41]:
torch.manual_seed(123)

optimizer = configure_optimizer(model, train_config.weight_decay, train_config.learning_rate, master_process= True, device=train_config.device)

num decayed parameter tensors: 253, with 3,085,697,024 parameters
num non-decayed parameter tensors: 181, with 241,664 parameters
using fused AdamW: True


In [42]:
train_comp = TrainComponents(optimizer=optimizer , dataloader=train_loader, scheduler= cosine_lr)
train_comp

TrainComponents(optimizer=AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.95)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: True
    lr: 1e-05
    maximize: False
    weight_decay: 0.1

Parameter Group 1
    amsgrad: False
    betas: (0.9, 0.95)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: True
    lr: 1e-05
    maximize: False
    weight_decay: 0.0
), dataloader=<model_training.framework.data.post_train.dataloader.CyclingDataLoader object at 0x1553ba007aa0>, scheduler=<function cosine_lr at 0x155385555080>)

In [43]:
torch.set_float32_matmul_precision('high') ## TF16 -> Model Training 

In [44]:
print(train_config.accumulation_steps)

2


In [45]:
model = train_model(model, train_components=train_comp, train_config=train_config)

Current Device: cuda
Model set to Training mode:True
step 0 | loss: 4.000000 | norm :390.00 | dt: 4959.38 ms | tok/sec:3303.64
step 1 | loss: 1.867188 | norm :146.00 | dt: 647.40 ms | tok/sec:25307.20


[W1101 03:01:20.004442260 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 14483456, total: 42436526080).


step 2 | loss: 1.390625 | norm :104.50 | dt: 862.36 ms | tok/sec:18999.08
step 3 | loss: 1.273438 | norm :102.00 | dt: 536.24 ms | tok/sec:30553.26


[W1101 03:01:21.427157292 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 14483456, total: 42436526080).


step 4 | loss: 0.949219 | norm :89.50 | dt: 828.01 ms | tok/sec:19787.16
step 5 | loss: 0.839844 | norm :63.50 | dt: 862.52 ms | tok/sec:18995.60


[W1101 03:01:23.139885213 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 3997696, total: 42436526080).


... (101 outputs left out)

[W1101 03:02:34.909950146 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 12386304, total: 42436526080).


step 113 | loss: 0.332031 | norm :4.41 | dt: 907.25 ms | tok/sec:18058.89
step 114 | loss: 0.431641 | norm :5.31 | dt: 530.95 ms | tok/sec:30858.02


[W1101 03:02:36.379208360 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 8192000, total: 42436526080).


step 115 | loss: 0.363281 | norm :5.16 | dt: 914.86 ms | tok/sec:17908.75


In [47]:
# text_1 = "What is your name ? "
text_1 = "Every effort moves you"
in_tokens = text_to_token_ids(text_1, tokenizer=tokenizer, 
                            #   allowed_special = {"<|endoftext|>"}
                              )
in_tokens=in_tokens.to(device)
print(in_tokens.shape)  # [1, sequence_length]


out_tokens = gen_sample(model, in_tokens, max_new_tokens=40, context_len=1024)
print(token_ids_to_text(out_tokens, tokenizer))

torch.Size([1, 4])
Every effort moves you closer toward an answer you don't yet understand.\nIn 1853, William Rowan Hamilton proposed the algebra of quaternions. It is not an algebraic system based on real


In [ ]:
# accelerator = Accelerator(mixed_precision="bf16")
# model, optimizer, loader = accelerator.prepare(model, optimizer, loader)